In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read Raw CSV from Volume

In [0]:
log("Reading raw CSV from Volume ...")

df_raw =(
    spark.read
    .option("header", "true")
    .option("interSchema", "true")
    .csv(RAW_FILE)
)

log(f"Raw rows loaded: {df_raw.count():,}")
df_raw.printSchema()

## 2. Add Ingestion Metadata

In [0]:
df_bronze = df_raw.withColumn("ingested_at", F.current_timestamp()) \
                .withColumn("source_file", F.col("_metadata.file_path")) \
                .withColumn("file_name", F.col("_metadata.file_name")) \
                .withColumn("file_size", F.col("_metadata.file_size"))

## 3. Rename Columns Before Writing to Bronze

In [0]:
# Rename all columns — replace spaces with underscores
# Converts all column names to snake_case
df_bronze = df_bronze.toDF(*[c.replace(" ", "_").replace("-", "_").lower() 
                          for c in df_bronze.columns])

log("✅ Columns renamed to snake_case.")
df_bronze.printSchema()

## 4. Write to Bronze Delta Table

In [0]:
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.raw_superstore")

In [0]:
# write_delta(df_bronze, TBL_BRONZE_RAW, mode="overwrite", cdf=False)